# Section 2: Data cleaning and preprocessing

We prepare the data to predict `repeat_purchase_90d`. We use the findings from `Exploration_notebook.ipynb` without repeating its analysis.

### What we already know

| Finding | What we will do |
|---|---|
| 6,534 purchases from 610 customers. | Keep purchase IDs and split by date. |
| 65 rows have almost no purchase information. | Keep the rows and fill missing inputs. |
| 603 first purchases lack several history values. | Mark first purchases separately from unknown history. |
| Some values are negative or reach one billion. | Replace invalid cells with missing values. |
| Some amounts and counts are much larger than most others. | Keep valid large purchases and use a log transformation. |
| Categories have no numerical order. | Use one-hot encoding. |
| Some variables are strongly correlated. | Leave feature selection to the next section. |
| The customer table has empty fields and many identifying labels. | Decide whether to use it. |

### Plan

Clean invalid values. Add a few features. Fill missing values. Encode categories and scale numbers where needed.

We fit preprocessing on earlier training purchases and check it on later ones. We also prepare test using settings learned from training only.

Model training, feature selection and prediction scores come later.

## 1. Load training data

We reload `train.csv` so this notebook runs on its own. `ID` stays as the index to match predictions to purchases later.

The folder setup works from the project root or `notebooks/`.

In [45]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

project_folder = Path.cwd()
if not (project_folder / "data").exists():
    project_folder = project_folder.parent

data_folder = project_folder / "data"
train = pd.read_csv(data_folder / "train.csv", index_col="ID")

## 2. Separate inputs and target

`X` holds the input columns. `y` holds `repeat_purchase_90d`, the answer we want to predict. The target must not be an input.

In [46]:
target = "repeat_purchase_90d"

X = train.drop(columns=target)
assert train[target].notna().all()
assert set(train[target].unique()) == {0, 1}
y = train[target].astype(int)

Exploration found 2,880 non-repeat purchases and 3,654 repeat purchases. Both groups are common, so we do not change their balance.

The checks above confirm that every target is present and is either 0 or 1.

## 3. Decide how to handle missing values

Exploration found **65 almost-empty purchases**, **603 first purchases** and **ten matching rows when ID is ignored**.

We keep them. Their targets are available. Matching values do not prove that different IDs refer to the same purchase.

We add flags before filling missing values. A flag is a 0/1 column that records missing information. We also mark first purchases separately: no previous purchases is different from an unknown purchase history.

## 4. Replace invalid values

We treat negative amounts, counts and durations as invalid for this purchase data. We replace these cells with `NaN`, meaning missing. We do not delete the rows. Customer IDs and `random_noise` are excluded from this rule.

We also replace values of at least one billion in `prior_mean_purchase_value` and `spend_365d`. Exploration found these likely errors. Infinite values become missing too.

Other large purchases stay. A high value alone does not make a purchase wrong.

In [47]:
categorical_columns = [
    "payment_type",
    "salesperson",
    "commercial_zone",
    "transport_method",
    "warehouse",
    "document_series",
]

# Exclude the customer identifier and the additional random field from this numeric group.
numeric_columns = X.select_dtypes(include="number").columns.tolist()
numeric_columns.remove("customer_id")
numeric_columns.remove("random_noise")


def clean_values(data):
    """Return a copy with invalid numeric cells replaced by missing values."""
    cleaned = data.copy()

    for column in numeric_columns:
        cleaned[column] = cleaned[column].replace([np.inf, -np.inf], np.nan)
        cleaned.loc[cleaned[column] < 0, column] = np.nan

    for column in ["prior_mean_purchase_value", "spend_365d"]:
        cleaned.loc[cleaned[column] >= 1_000_000_000, column] = np.nan

    return cleaned

The function cleans a copy of the input. It keeps all rows. The new gaps are filled later, together with the original missing values.

## 5. Decide whether to use `clientes.csv`

Exploration showed that each customer has one row in this table and every training purchase has a match. We also found that `customer_type` is empty. Company names and addresses mostly identify customers.

We leave this table out for now. We do not know whether its information was available at each purchase date. A later customer update could give the model information from the future.

If this is clarified, we can test selected fields. We would use a left join on `customer_id` to keep every purchase. Each purchase must match at most one customer row.

## 6. Add features and remove some columns

| Feature | What it tells us |
|---|---|
| `first_purchase` | The known previous purchase count is zero. |
| `history_unknown` | The previous purchase count is missing. |
| `..._missing` | An original numeric value was missing or invalid. |
| `net_value_per_detail` | Net purchase value divided by the number of details. |

We only divide by a positive detail count. Otherwise, value per detail stays missing until filling. We also remove spaces around category labels to avoid treating the same label as different categories.

We remove `customer_id` because it is an identifier. We use `purchase_date` for splitting, but not as a model input. We leave calendar features out because the dates are anonymized.

We leave `random_noise` out for this first version. Exploration found little relationship with the target. That does not prove it is useless. Feature selection should test this choice and review the correlated business variables.

In [39]:
def add_features(data):
    """Clean the inputs and add simple features without learning statistics."""
    prepared = clean_values(data)

    # Record missing information before the gaps are filled.
    for column in numeric_columns:
        prepared[column + "_missing"] = prepared[column].isna().astype(int)

    prepared["first_purchase"] = (prepared["prior_purchase_count"] == 0).astype(int)
    prepared["history_unknown"] = prepared["prior_purchase_count"].isna().astype(int)

    # A zero or missing detail count cannot be used as a divisor.
    detail_count = prepared["N_ Detalhes"].where(prepared["N_ Detalhes"] > 0)
    prepared["net_value_per_detail"] = prepared["Total Liquido"] / detail_count

    # Strip accidental spaces from category names. Keep missing values as NaN.
    for column in categorical_columns:
        labels = prepared[column].astype("string").str.strip()
        labels = labels.replace("", pd.NA)
        prepared[column] = labels.to_numpy(dtype=object, na_value=np.nan)

    # ID is already the index. Keep the date in X for splitting, but not as a feature.
    prepared = prepared.drop(columns=["customer_id", "purchase_date", "random_noise"])
    return prepared


feature_preview = add_features(X)
preview_columns = [
    "prior_purchase_count", "first_purchase", "history_unknown",
    "Total Liquido", "N_ Detalhes", "net_value_per_detail",
]
display(feature_preview[preview_columns].head())

,prior_purchase_count,first_purchase,history_unknown,Total Liquido,N_ Detalhes,net_value_per_detail
ID,,,,,,
10000000,2.0,0,0,453.75,4.0,113.437500
10000001,5.0,0,0,795.77,32.0,24.867812
10000002,90.0,0,0,8.40,1.0,8.400000
10000003,1.0,0,0,696.61,9.0,77.401111
10000004,25.0,0,0,1313.90,33.0,39.815152


The preview shows the added features before any log transformation or scaling. `net_value_per_detail` is `Total Liquido` divided by `N_ Detalhes`.

Flags use 1 for yes and 0 for no. A missing purchase count gives `history_unknown = 1` and `first_purchase = 0`. We cannot call it a first purchase when we do not know its history.

## 7. Split earlier and later purchases

Test purchases come later than training purchases. We use roughly the last 20% of dated training purchases as validation. Purchases on the same date stay together.

We leave a gap of more than 90 days before validation. This gives the earlier targets time to become known. Rows without dates are left out of this check only.

This split checks preprocessing. Model comparison comes later.

In [40]:
purchase_dates = pd.to_datetime(X["purchase_date"], errors="coerce")
sorted_dates = purchase_dates.dropna().sort_values()

split_position = int(len(sorted_dates) * 0.80)
validation_start = sorted_dates.iloc[split_position]
fitting_end = validation_start - pd.Timedelta(days=90)

fitting_rows = purchase_dates < fitting_end
validation_rows = purchase_dates >= validation_start
gap_rows = purchase_dates.notna() & ~fitting_rows & ~validation_rows

X_fit = X.loc[fitting_rows].copy()
X_validation = X.loc[validation_rows].copy()
y_fit = y.loc[fitting_rows]
y_validation = y.loc[validation_rows]

assert not X_fit.empty and not X_validation.empty
assert purchase_dates[fitting_rows].max() + pd.Timedelta(days=90) < validation_start

print("Validation begins:", validation_start.date())
print("Fitting purchases:", len(X_fit))
print("Validation purchases:", len(X_validation))
print("Purchases in the gap:", gap_rows.sum())
print("Purchases without a date:", purchase_dates.isna().sum())

Validation begins: 2023-06-17
Fitting purchases: 5024
Validation purchases: 1298
Purchases in the gap: 147
Purchases without a date: 65


Validation starts on **17 June 2023**. We have **5,024 fitting rows**, **1,298 validation rows**, **147 gap rows** and **65 rows without dates**. None are deleted from the full training data.

A customer can appear in both periods. We want to predict future purchases by existing customers too. Exploration already used the full training data, so this is not a completely untouched final check.

## 8. Fill missing values

We fill numeric gaps with the fitting median after the log transformation. The median is less affected by very large values than the mean.

Missing categories become `__MISSING__`. Numeric missingness flags remain so the model can see which inputs were missing.

Validation and test do not help calculate the replacements.

## 9. Encode categories

We use one-hot encoding. Each known category gets a separate 0/1 column. This avoids giving labels a made-up numerical order.

The category list comes from fitting rows only. `handle_unknown="ignore"` accepts new labels in validation or test. Their category block contains zeros because there is no fitted column for that label.

## 10. Transform and scale numbers

`log1p` means log of (1 + value). We apply it to nonnegative amounts, counts and durations, including value per detail. It reduces large differences and keeps zero valid. We do not log the 0/1 flags.

After filling gaps, `StandardScaler` uses the fitting mean and standard deviation to put numeric inputs on similar scales. This helps models such as logistic regression and nearest neighbours. Numeric flags are scaled too. One-hot columns stay at 0/1.

For trees, `build_preprocessor(scale=False)` skips scaling. It still applies the log transformation. Modelling must check whether these choices help.

In [41]:
# These are the nonnegative magnitude columns; missing flags are not included.
log_columns = numeric_columns + ["net_value_per_detail"]


def prepare_features(data):
    prepared = add_features(data)

    for column in log_columns:
        prepared[column] = np.log1p(prepared[column])

    return prepared


prepared_example = prepare_features(X_fit)
model_numeric_columns = prepared_example.select_dtypes(include="number").columns.tolist()

print("Numeric features, including flags:", len(model_numeric_columns))
print("Categorical features before encoding:", len(categorical_columns))

Numeric features, including flags: 59
Categorical features before encoding: 6


We now have **59 numeric features** and **six categorical features** before encoding. The numeric group includes the new flags and value per detail.

So far, we have only used fixed rules. No medians, category lists or scaling statistics have been fitted.

## 11. Put the steps in one pipeline

The pipeline runs the same steps each time:

1. Clean values, add features and apply the log transformation.
2. Fill numeric gaps and optionally scale numbers.
3. Fill categorical gaps and encode labels.
4. Combine the results.

`ColumnTransformer` gives each column group its own steps. The full pipeline can be fitted again inside each training fold.

In [42]:
def build_preprocessor(scale=True):
    # Route 1: numeric columns.
    numeric_steps = [
        ("fill_missing", SimpleImputer(strategy="median", keep_empty_features=True)),
    ]
    if scale:
        numeric_steps.append(("scale", StandardScaler()))

    numeric_pipeline = Pipeline(numeric_steps)

    # Route 2: categorical columns.
    categorical_pipeline = Pipeline([
        ("fill_missing", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
        ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])

    combine_columns = ColumnTransformer([
        ("numeric", numeric_pipeline, model_numeric_columns),
        ("categorical", categorical_pipeline, categorical_columns),
    ])

    # Clean/create features first, then run the two routes.
    return Pipeline([
        ("prepare_features", FunctionTransformer(prepare_features)),
        ("combine_columns", combine_columns),
    ])

Each call creates a new preprocessor. It learns its settings when fitted.

If a numeric column has no observed fitting values, `keep_empty_features=True` keeps it and fills it with zero. We use a regular output array because this dataset is small enough.

## 12. Check fitting and validation outputs

`fit_transform` learns settings from fitting rows and applies them. `transform` applies the same settings to validation without learning anything new.

We check that both outputs have matching columns and no missing or infinite values.

In [43]:
preprocessor = build_preprocessor()

X_fit_prepared = preprocessor.fit_transform(X_fit)
X_validation_prepared = preprocessor.transform(X_validation)

print("Fitting table (rows, columns):", X_fit_prepared.shape)
print("Validation table (rows, columns):", X_validation_prepared.shape)
print("Fitting values are all finite:", np.isfinite(X_fit_prepared).all())
print("Validation values are all finite:", np.isfinite(X_validation_prepared).all())

assert X_fit_prepared.shape[1] == X_validation_prepared.shape[1]
assert np.isfinite(X_fit_prepared).all()
assert np.isfinite(X_validation_prepared).all()

# Show a few numeric columns instead of displaying the entire wide table.
column_names = preprocessor.named_steps["combine_columns"].get_feature_names_out()
prepared_preview = pd.DataFrame(
    X_fit_prepared[:3], columns=column_names, index=X_fit.index[:3]
)
display(prepared_preview.iloc[:, :5])

Fitting table (rows, columns): (5024, 164)
Validation table (rows, columns): (1298, 164)
Fitting values are all finite: True
Validation values are all finite: True


,numeric__Total a Pagar,numeric__Total Bruto,numeric__Total Liquido,numeric__Total Imp_,numeric__Total Portes
ID,,,,,
10000001,0.674879,0.606556,0.663172,0.739154,-0.455063
10000002,-2.627660,-2.476191,-2.616205,-2.146727,-0.455063
10000003,0.576436,0.476806,0.565005,0.646913,-0.455063


The outputs have **5,024 fitting rows** and **1,298 validation rows**, both with **164 features**. All values are finite.

The preview shows logged and scaled values. A negative number means below the fitting mean on that scale. It does not mean the original purchase amount was negative.

The data is ready for a model. This does not tell us how good its predictions will be.

## 13. Prepare all training and test inputs

We fit a new preprocessor on all training inputs. We then apply it to test using only `transform`. This shows how final preparation works once modelling choices are settled.

Rows without dates are included again. IDs are kept separately for matching predictions later. We do not train a classifier or create a submission here.

In [44]:
test = pd.read_csv(data_folder / "test.csv", index_col="ID")
assert set(test.columns) == set(X.columns)
assert train.index.is_unique and test.index.is_unique

final_preprocessor = build_preprocessor()
X_all_prepared = final_preprocessor.fit_transform(X)
X_test_prepared = final_preprocessor.transform(test[X.columns])

# Keep IDs alongside the prepared inputs for future predictions.
train_ids = X.index.copy()
test_ids = test.index.copy()

print("All training inputs (rows, columns):", X_all_prepared.shape)
print("Test inputs (rows, columns):", X_test_prepared.shape)
print("All training values are finite:", np.isfinite(X_all_prepared).all())
print("All test values are finite:", np.isfinite(X_test_prepared).all())

assert len(X_all_prepared) == len(X)
assert len(X_test_prepared) == len(test)
assert X_all_prepared.shape[1] == X_test_prepared.shape[1]
assert np.isfinite(X_all_prepared).all()
assert np.isfinite(X_test_prepared).all()

All training inputs (rows, columns): (6534, 173)
Test inputs (rows, columns): (986, 173)
All training values are finite: True
All test values are finite: True


The final outputs have **6,534 training rows** and **986 test rows**, both with **173 features**. All purchases are kept. There are no missing or infinite values.

The full training data has more categories than the earlier fitting period. That is why the feature count increased. Test did not add categories to the fitted list.

## 14. Conclusion

We used the Exploration findings to prepare the data for modelling:

- Replaced invalid values with missing values instead of deleting purchases.
- Filled numeric gaps with training medians and missing categories with `__MISSING__`.
- Added missingness and first-purchase flags so filling gaps would not hide missing history.
- Added value per detail to describe purchase value relative to its size.
- Removed identifiers from model inputs and left out `clientes.csv` because its information might have been updated after the purchase.
- Encoded categories as 0/1 columns. Applied a log transformation to reduce large numeric differences and added optional scaling for models that need it.

All 6,534 training purchases and 986 test purchases were kept in the final prepared data, with no missing or infinite values. The pipeline repeats the same steps each time and learns its settings from training data only.
